# Astra v4 — Dedicated Chat Training from Scratch (T4, ~30 min)

Trains a standalone conversational model **v4** for chat separately, using the best word-level data without depending on any external checkpoints.

**Pipeline (2 stages, 1 tokenizer):**

| Stage | Config | Data | Steps | Output |
|---|---|---|---|---|
| 0 | — | prose + chat | — | `tokenizer/artifacts/prose_chat_word.json` |
| 1 | `configs/astra5m_word_prose.json` | `datasets/prose/train.txt` | 5400 | `/content/runs/v4_prose/final.npz` |
| 2 | `configs/astra5m_word_chat.json` | `datasets/chat/train.txt` | 3800 | `/content/runs/v4_chat/resumed/final.npz` |

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
import os, subprocess
REPO = '/content/astra'
if os.path.isdir(REPO):
    subprocess.run(f'rm -rf {REPO}', shell=True, check=True)
subprocess.run(f'git clone --depth 1 https://github.com/anoneurx/astra.git {REPO}', shell=True, check=True)
assert os.path.isdir(REPO), 'clone failed'
os.chdir(REPO)
print('repo ready at', os.getcwd())

## STAGE 0 — Build Word-Level Tokenizer

In [ ]:
import os, subprocess
os.chdir('/content/astra')
if not os.path.exists('tokenizer/artifacts/prose_chat_word.json'):
    subprocess.run('python tools/build_word_tokenizer.py', shell=True, check=True)
print('Tokenizer ready.')

## STAGE 1 — Base Prose Training

In [ ]:
import os, subprocess, glob, shutil
os.chdir('/content/astra')
os.makedirs('/content/runs/v4_prose', exist_ok=True)
subprocess.run('python training/gpu_train.py --config configs/astra5m_word_prose.json --out /content/runs/v4_prose', shell=True, check=True)
hits = glob.glob('/content/runs/v4_prose/**/final.npz', recursive=True)
if hits and not os.path.exists('/content/runs/v4_prose/final.npz'):
    shutil.copy(hits[0], '/content/runs/v4_prose/final.npz')
print('Stage 1 complete. Final weights:', '/content/runs/v4_prose/final.npz')

## STAGE 2 — Chat Fine-Tuning (v4)

In [ ]:
import os, subprocess, glob, shutil
os.chdir('/content/astra')
if not os.path.exists('/content/runs/v4_prose/final.npz'):
    hits = glob.glob('/content/runs/v4_prose/**/final.npz', recursive=True)
    if hits:
        os.makedirs('/content/runs/v4_prose', exist_ok=True)
        shutil.copy(hits[0], '/content/runs/v4_prose/final.npz')
assert os.path.exists('/content/runs/v4_prose/final.npz'), 'Base prose final not found! Run Stage 1 first.'
os.makedirs('/content/runs/v4_chat', exist_ok=True)
subprocess.run('python training/gpu_train.py --config configs/astra5m_word_chat.json --out /content/runs/v4_chat --resume /content/runs/v4_prose/final.npz', shell=True, check=True)
chat_hits = glob.glob('/content/runs/v4_chat/**/final.npz', recursive=True)
if chat_hits and not os.path.exists('/content/runs/v4_chat/final.npz'):
    shutil.copy(chat_hits[0], '/content/runs/v4_chat/final.npz')
print('Stage 2 complete. Chat final weights:', '/content/runs/v4_chat/final.npz')

## STAGE 3 — Inference & Chat Testing

In [ ]:
import sys, os, glob, json, numpy as np
sys.path.insert(0, '/content/astra/python')
from astra.inference import KVCache, decode
from astra.learning.evaluate import load_model
from astra.model import ModelConfig
from astra.tokenizer.word import WordLevel

os.chdir('/content/astra')
CANDIDATES = sorted(glob.glob('/content/runs/v4_chat/**/final.npz', recursive=True))
CKPT = CANDIDATES[0] if CANDIDATES else None
if not CKPT and os.path.exists('/content/runs/v4_chat/final.npz'):
    CKPT = '/content/runs/v4_chat/final.npz'
assert CKPT, 'NO CHAT FINAL FOUND'

cfg = json.load(open('configs/astra5m_word_chat.json'))
tok = WordLevel.load(cfg['tokenizer'])
mcfg = ModelConfig.from_dict(cfg['model']); mcfg.vocab_size = len(tok)
model = load_model(mcfg, CKPT)
cache = KVCache(mcfg)
history = []
SPECIALS = set(range(tok.num_special))

def ask(user_text, max_new=48, temperature=0.7, top_k=40, rep_penalty=1.15):
    global history
    history.append(f'You: {user_text}')
    prompt = '\n'.join(history) + '\nAstra:'
    ids = tok.encode(prompt)[-(mcfg.max_seq_len - max_new):]
    new = decode(model, ids, max_new=max_new, temperature=temperature,
                 rng=np.random.default_rng(len(history)), top_k=top_k,
                 cache=cache, rep_penalty=rep_penalty, forbidden=SPECIALS)
    reply = tok.decode(new).split('\nYou:')[0].strip()
    history.append(f'Astra: {reply}')
    return reply

for q in ['hello', 'what is coding', 'tell me a joke']:
    print('You:', q)
    print('Astra:', ask(q), '\n')